In [2]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source":"mammal-pets-doc"},
    ),

    Document(
       page_content = "Cats are independent pets that often enjoy their own space",
       metadata={"source":"mammal-pets-doc"}
     ),

     Document(
       page_content = "Goldfish are popular pets for begineers, requiring relatively simple care.",
       metadata={"source":"mammal-pets-doc"}
     ),

     Document(
       page_content = "Parrots are intelligent birds capable of mimicking human speech",
       metadata={"source":"mammal-pets-doc"}
     ),
     Document(
         page_content="Rabbits are social animals that need plenty of space to hop around.",
         metadata={"source":"mammal-pets-docs"}
     )
]

In [3]:
documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Goldfish are popular pets for begineers, requiring relatively simple care.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech'),
 Document(metadata={'source': 'mammal-pets-docs'}, page_content='Rabbits are social animals that need plenty of space to hop around.')]

In [5]:
type(documents)

list

In [8]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

llm = ChatGroq(groq_api_key=groq_api_key,model="llama-3.1-8b-instant")

In [10]:
## Embedding Techniques using Hugging face
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1629.86it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
## Vector Store 
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(documents,embedding=embeddings)
vectorstore


In [13]:
vectorstore.similarity_search("cat")

[Document(id='3c4a82c6-7164-4b6d-a685-b45eca5df5ac', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space'),
 Document(id='f86e6a66-7735-469e-9f49-248d8a7db5fd', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space'),
 Document(id='84ad53f9-ef39-4c00-bb68-871db6c16e73', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='c67b0d19-f3cf-4c45-bf52-356e5c73fa56', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]

In [14]:
## Async Query
await vectorstore.asimilarity_search("cat")

[Document(id='3c4a82c6-7164-4b6d-a685-b45eca5df5ac', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space'),
 Document(id='f86e6a66-7735-469e-9f49-248d8a7db5fd', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space'),
 Document(id='84ad53f9-ef39-4c00-bb68-871db6c16e73', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='c67b0d19-f3cf-4c45-bf52-356e5c73fa56', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]

In [15]:
vectorstore.similarity_search_with_score("cat")

[(Document(id='3c4a82c6-7164-4b6d-a685-b45eca5df5ac', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space'),
  0.9436444044113159),
 (Document(id='f86e6a66-7735-469e-9f49-248d8a7db5fd', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space'),
  0.9436444044113159),
 (Document(id='84ad53f9-ef39-4c00-bb68-871db6c16e73', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.574089765548706),
 (Document(id='c67b0d19-f3cf-4c45-bf52-356e5c73fa56', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.574089765548706)]

In [16]:
## Retrievers
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriever.batch(["cat","dog"])

[[Document(id='3c4a82c6-7164-4b6d-a685-b45eca5df5ac', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space')],
 [Document(id='c67b0d19-f3cf-4c45-bf52-356e5c73fa56', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

In [19]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kawrgs={"k":1}
)

retriever.batch(["cat","dog"])


[[Document(id='3c4a82c6-7164-4b6d-a685-b45eca5df5ac', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space'),
  Document(id='f86e6a66-7735-469e-9f49-248d8a7db5fd', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space'),
  Document(id='84ad53f9-ef39-4c00-bb68-871db6c16e73', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  Document(id='c67b0d19-f3cf-4c45-bf52-356e5c73fa56', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')],
 [Document(id='84ad53f9-ef39-4c00-bb68-871db6c16e73', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  Document(id='c67b0d19-f3cf-4c45-bf52-356e5c73fa56', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are gr

In [23]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer the question using the provided context only
{question}

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([("human",message)])

rag_chain ={"context":retriever,"question":RunnablePassthrough()} | prompt | llm

response = rag_chain.invoke("tell me about dogs")
print(response.content)

Dogs are great companions, known for their loyalty and friendliness.
